In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# ----------------------------------------
# Single Variate Linear Regression
# ----------------------------------------

from sklearn.datasets import fetch_california_housing

# Load the California housing dataset
hd = fetch_california_housing()

# Convert the dataset into a pandas DataFrame
housing = pd.DataFrame(hd.data, columns=hd.feature_names)

# Store the target values separately
target = hd.target

# Plot one feature against the target to inspect the relationship
housing.iloc[50:100].plot.scatter(x='MedInc', y=target[50:100])
plt.xlabel('median income in block group')
plt.ylabel('Median House Value x100,000')
plt.show()

# Display the dataset description
print(hd.DESCR)

# Show the first few rows of the feature table
housing.head()

# Select 50 rows from the dataset for easier visualization and training
data = housing.loc[50:100].to_numpy()

# Normalize all feature columns except the target column
# This uses max normalization over each feature column
data[:, :-1] = data[:, :-1] / data[:, :-1].max(axis=0)

# Choose one predictor variable: MedInc
x = data[:, 0]

# Choose the response variable: target
y = data[:, -1]

# Reshape x into a 2D array for matrix operations
x.shape

# ----------------------------------------
# Gradient Descent Utilities
# ----------------------------------------

def gradient(t, X, y):
    # Compute predicted y values using the current parameters
    yestimate = X.dot(t.flatten())

    # Compute the residuals between predictions and actual values
    loss = yestimate - y.flatten()

    # Compute the gradient of the mean squared error cost function
    grad = 1.0 / X.shape[0] * X.T.dot(loss)

    # Compute the cost value for reporting
    cost = float(0.5 * np.mean((yestimate - y.flatten()) ** 2))

    # Return gradient in the same shape as t and the cost
    return grad.reshape(t.shape), cost

def computecost(t, X, y):
    # Compute the predicted values
    yestimate = X.dot(t.flatten())

    # Return the mean squared error cost
    return float(0.5 * np.mean((yestimate - y.flatten()) ** 2))

def gradientdescent(x, y, alpha=0.5, tolerance=1e-5, maxit=1e6, nulbias=False):
    # Ensure x is a column vector
    if x.ndim == 1:
        x = x.reshape(-1, 1)

    # Add a bias column of ones
    X = np.hstack([np.ones((x.shape[0], 1)), x])

    # Initialize parameters randomly
    t = np.random.randn(X.shape[1], 1)

    # Optionally force the bias term to zero
    if nulbias:
        t[0] = 0

    it = 0
    models = []
    grads = []
    errors = []

    # Iterate until convergence or max iterations
    while it < maxit:
        grad, error = gradient(t, X, y)

        # Save the optimization history
        models.append(t.copy())
        grads.append(grad.copy())
        errors.append(error)

        # Update parameters using gradient descent
        newt = t - alpha * grad

        # Keep bias term fixed at zero if requested
        if nulbias:
            newt[0] = 0

        # Stop if parameters barely change
        if np.sum(abs(newt - t)) < tolerance:
            break

        t = newt
        it += 1

    if it == maxit:
        print("Warning: reached maximum number of iterations without convergence.")

    return X, t, models, grads, errors

def plotmodel(x, y, t, startatzero=False):
    # Plot the data and a fitted line
    if t is not None:
        if startatzero:
            x = np.append(0, x)
            y = np.append(0, y)

        plt.plot(x, t[0] + t[1] * x, 'g', label='Model')
        plt.scatter(x, y, c='b', label='Data')
        plt.legend(loc='best')
        plt.xlabel('MedInc')
        plt.ylabel('Median House Price x100,000')

        if startatzero:
            plt.ylim(ymin=0)
            plt.xlim(xmin=0)

        plt.show()

# Plot the raw data
plotmodel(x, y, None)
plotmodel(x, y, np.array([4, 0]))

# Run gradient descent without a bias term
X, t, models, grads, errors = gradientdescent(x, y, nulbias=True)

print("iterations", len(models))
print("first model", models[0])
print("best model", t)

# Plot several models along the optimization path
nits = len(models)
ts = [0, nits // 4, nits // 2, 3 * nits // 4, nits - 1]

for i in ts:
    print("Iteration", i + 1)
    print("Gradient", grads[i])
    print("Error", errors[i])
    plotmodel(x, y, models[i])

plotmodel(x, y, models[-1], True)

# Plot the cost values over time
plt.plot([m[1] for m in models], errors, 'b--', marker='o', markersize=12, markeredgecolor='r')
plt.xlabel('r')
plt.ylabel('J')
plt.show()

# Run gradient descent with a bias term
X, t, models, grads, errors = gradientdescent(x, y)

print("iterations", len(models))
print("first model", models[0])
print("best model", t)

nits = len(models)
ts = [0, nits // 4, nits // 2, 3 * nits // 4, nits - 1]

for i in ts:
    print("Iteration", i + 1)
    print("Gradient", grads[i])
    print("Error", errors[i])
    plotmodel(x, y, models[i])

# ----------------------------------------
# Cost Surface Visualization
# ----------------------------------------

from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D

def plotcost(X, y, start=-100, end=100, step=2, rotate=False):
    # Create a 3D plot of the cost function
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    t0s = []
    t1s = []
    errors = []

    # Evaluate the cost function over a grid of parameter values
    for t0 in np.arange(start, end, step):
        for t1 in np.arange(start, end, step):
            t0s.append(t0)
            t1s.append(t1)
            errors.append(computecost(np.array([t0, t1]), X, y))

    # Plot the 3D surface
    surf = ax.plot_trisurf(t0s, t1s, errors, cmap=cm.jet, linewidth=0.2)
    fig.colorbar(surf)

    plt.xlabel('t0')
    plt.ylabel('t1')

    if rotate:
        ax.view_init(30, 30)

    fig.tight_layout()
    plt.show()

plotcost(X, y, start=-50, end=70)
plotcost(X, y, start=-50, end=70, rotate=True)

# ----------------------------------------
# Compare with scikit-learn
# ----------------------------------------

from sklearn.linear_model import LinearRegression

# Fit a linear regression model without intercept
modelsk = LinearRegression(fit_intercept=False)

# Generate synthetic data for comparison
np.random.seed(42)
nsamples = data.shape[0]
X = np.random.rand(nsamples, 2)
truecoef = np.array([2.0, -3.5])
y = X @ truecoef + np.random.randn(nsamples) * 0.9

# Fit the sklearn model
modelsk.fit(X, y)

# Fit the gradient descent model
X2, t, models, grads, errors = gradientdescent(X[:, 0], y, nulbias=True, alpha=0.01, maxit=100000)

print("Sklearn Coefficients", modelsk.coef_)
print("GD Coefficients", t[1])
print("Sklearn Intercept", modelsk.intercept_)
print("GD Intercept", t[0])
print("Sklearn Error", float(0.5 * np.mean((modelsk.predict(X) - y.flatten()) ** 2)))
print("GD Error", float(0.5 * np.mean((X2 @ t.flatten() - y.flatten()) ** 2)))

## Task: Multi-Variate Linear Regression

Modify the gradient function given to you earlier to perform multivariate linear regression, i.e., the input contains more than one variable. Then, use the function you created to find the combination of any two dependent variables with the least root mean square error with regards to the response/target variable. You do not need to plot the cost function for this exercise.

In [ ]:
# get column names
Index = housing.columns[:-1]

# extract 50 rows
data = housing.loc[51:100].to_numpy()
# normalize the data except for the target
data[:, :-1] = data[:, :-1] / data[:, :-1].max(axis=0)

# get the target
y = data[:,-1].reshape(-1, 1)
results = []
for i in range(len(housing.columns)-1):
  x1 = data[:,i]
  for j in range(i+1, len(housing.columns)-1):
    x2 = data[:,j]
    ...
# get results with the lowest error
results = sorted(results, key=lambda x: x[2])
print("Best pair of features: ", results[0])

Repeat the same exercise using sklearn's linear regression model.

Compare the results and discuss your findings.